# Whiteboard Studio · Tạo video vẽ tay tiếng Việt

Notebook này chạy source riêng trong repo của bạn, không clone Chalkboard. Nhập chủ đề, xem kịch bản và prompt theo từng cảnh, theo dõi âm thanh/video đang được dựng, rồi tải MP4 và file kịch bản.

Pipeline: **Chủ đề → Kịch bản + prompt cảnh → Piper đọc từng cảnh → Vẽ hoạt họa từng cảnh → Ghép thành MP4**.

Giao diện chạy ngay trong Colab. Lần đầu sẽ tải model Qwen2.5 3B (~1,9 GB), thư viện, và giọng đọc Piper tiếng Việt.

## 1. Cài công cụ trong Colab

Runtime cần Linux. Cell này thêm FFmpeg để xuất MP4, zstd để bộ cài Ollama hoạt động, và font hỗ trợ tiếng Việt.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y curl ca-certificates ffmpeg zstd fonts-dejavu-core

## 2. Tải source của bạn

Đảm bảo các file `whiteboard_studio.py` và `requirements.txt` được tải lên thư mục gốc repo. Cell này lấy source từ GitHub repo của bạn, không lấy mã từ repo khác.

In [ ]:
import os
import subprocess
import sys

SOURCE_REPO = "https://github.com/saigonvet8312-coder/colab-whiteboard.git"
SOURCE_DIR = "/content/colab-whiteboard"

if os.path.isdir(os.path.join(SOURCE_DIR, ".git")):
    subprocess.run(["git", "-C", SOURCE_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", SOURCE_REPO, SOURCE_DIR], check=True)

%cd /content/colab-whiteboard
!python -m pip -q install -r requirements.txt

## 3. Cài và khởi động Ollama

Model nhỏ hơn để Colab dễ chạy. Có GPU T4 thì chọn GPU; nếu Ollama chưa dùng GPU, nó vẫn có thể chạy trên CPU nhưng chậm hơn.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

import time
import urllib.request

try:
    urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
    print("Ollama đã chạy.")
except Exception:
    OLLAMA_LOG = open("/tmp/ollama.log", "w")
    OLLAMA_PROCESS = subprocess.Popen(
        ["ollama", "serve"],
        stdout=OLLAMA_LOG,
        stderr=subprocess.STDOUT
    )
    for _ in range(60):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
            break
        except Exception:
            time.sleep(2)
    else:
        raise RuntimeError("Ollama chưa khởi động. Xem /tmp/ollama.log")

!ollama pull qwen2.5:3b
os.environ["OLLAMA_MODEL"] = "qwen2.5:3b"

## 4. Tải giọng đọc tiếng Việt

Piper tạo file WAV riêng cho từng cảnh. Giọng `vi_VN-vais1000-medium` được tải vào máy ảo Colab.

In [ ]:
!mkdir -p /content/piper-voices
!bash -lc "cd /content/piper-voices && python -m piper.download_voices vi_VN-vais1000-medium"

os.environ["PIPER_MODEL"] = "/content/piper-voices/vi_VN-vais1000-medium.onnx"

## 5. Mở giao diện tạo video

Chạy cell này để mở bảng điều khiển. Trong giao diện, nhập chủ đề rồi bấm **Tạo kịch bản và video**. MP4 và kịch bản JSON có nút tải riêng.

In [ ]:
%cd /content/colab-whiteboard
from whiteboard_studio import launch
launch()